# Model Architecture
In this notebook, we take a look how we model the diffusion framework in this repository, how the model architecture is built up, and what the different model blocks do. Furthermore, we discuss the design choices behind some of the other parts of diffusion, namely the noise scheduler, sampler, and optimizer.

## How is our diffusion model built up?
This repository mostly builds on the architecture of the paper [PixArt-Alpha: Fast Training of Diffusion Transformer for Photorealistic Text-to-Image Synthesis](https://arxiv.org/abs/2310.00426), which is, depspite its age, worth checking out due to its focus on practicality and many of their design choices being still being present in modern architectures and training routines. In any case, you can also just work through this notebook and you will have all information needed to understand what's going on in this code base. One important thing to mention is that we don't denoise images directly. Instead we denoise a so-called latent representation of it. Take a look at the diagram below. You see that we enter $x$, the image, into an encoder block before we apply diffusion to it and then decode the output again into an image at the end. The reason we do this is to make the computation more efficient. The idea is to train a model, i.e. a variational autoencoder (VAE), to compress an image into a smaller latent representation, so that it can still reconstruct it with high fidelity. A latent is often a magnitude smaller than the original image, while still showing no visible deterioration in aesthetics, and since diffusion is data-agnostic, we can train a model on denoising latents, instead of the image itself. This makes diffusion much more scalable and cheaper to train. Diffusion models that work in this way are called **Latent Diffusion Models (LDMs)** and were introduced in the paper [High-Resolution Image Synthesis with Latent Diffusion Models](http://arxiv.org/abs/2112.10752), authored by the team behind Stable Diffusion at Stability AI, which kicked off the current success wave of diffusion models.

![image](../img/ldm.jpg)

Another nice property of those models are that we can decouple training objectives here. Namely, we can train an image VAE on producing good latent representations and then use the frozen weights of its encoder and decoder when training the backbone (that's how the part that does the prediction for the denoising is commonly referred to) of the LDM. Training VAEs is a stand-alone topic that isn't the focus of this repo, hence we will concentrate from now on on the backbone. The backbone we use is a modified version of the Diffusion Transformer, introduced in the paper [Scalable Diffusion Models with Transformers](http://arxiv.org/abs/2212.09748), that introduced an attention-based architecture that was inspired by vision transformers and leaned on the success for the transformer architecture in various domains. In the next few sections, we will go through each part of of the backbone to give an idea what they do.

![image](../img/dit_architecture.png)



### A Very Short Explanation of the Attention Mechanism
The attention mechanism is at the heart of the transformer architecture and understanding its core principle, makes understanding some of the other blocks much easier. The attention mechanism is a dynamic routing algorithm that tries to find tokens relevant to each other and exchange information between them. It does this by creating a key ($K$), value ($V$), and query ($Q$) vector for each token via linear projection. Afterwards, for each token, the $Q$ of the original token is multiplied (inner product) with the $K$ of all tokens to detemine their weight/importance for the creation of the next state of the original token. In the next step, we use those importance weights to create a weighted average of the $V$s of all tokens. This value is then brought again into the original token dimension with a linear projection. An intuitive way to think about those vectors is to imagine $K$ as the vector representing what kind of information a token contains, $Q$ as what the token is looking for, and $V$ as the actual information. $Q$ of a token searches via vector similarity after relevant information in the other tokens. 

![image](../img/attention.png)

When you check out the figures and follow the path through which a token is processed, you will see that the attention blocks are the only way how information between the tokens is exchanged. Embedding, hifting and scaling, as well as the feedforward networks are applied on each token in isolation. So all the magic is actually happening in the attention blocks.


This was a really superficial explanation of the attention mechanism and by no means comrehensive. If you want a deeper intuition of the transformer architecture, I recommend the article [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/).

### Patchify & Reshape
Latent representations that we receive from an encoder are usually still in a image-like form, i.e. [width x height x channel], and we have to convert them into a sequence tokens so the transformer's attention mechanism can do its magic. You can see how we do this in the figure below (although in our code we are doing it on a latent instead of an image, the procedure stays the same). We start by breaking up the image in patches. The patches are then flattened and transformed into the right dimension via a linear projection.

![image](../img/patch_embedding.jpg)

### Label Embedding
When we use our diffusion model to generate an image, we want to be able to control the subject. This is where the label comes in. In this project, we work with the CIFAR-10 dataset, which images each categorized as one of ten different classes. We use those labels for training the model to understand the connection between them and the images it generates. But for this, we have to bring the class labels in a form, so that the transformer architecture can incorporate them, which we call embedding. Label embedding is done with PyTorch's [Embedding module](https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html). It holds a vector of the same dimension as the embedded patches for each class whose final values will be learned during training. We will see in a later section how the embedding will be used in the attention block.


You may be wondering how conditioning would work when we don't have a text prompt instead of a simple class label. In this case, we would remove the prediction head of an LLM and use the last-layer token embeddings for your conditioning. In case that the token dimensions of the LLM and diffusion backbone are different, one would adjust the LLM's token dimension with a linear projection.  

### Timestep Embedding
For the timestep, we chose a fixed representation via sinuoidal positional encoding. Sinusoidal encoding has some advantageous properties for the representation of time, which will be explained in more detail in the [Multi-Head Attention](#multi-head-attention) section. You can find the code implementation in the class `TimeEmbedding` in `diffusion/modules.py`.


### Diffusion Transformer (DiT) Block
In case you're aquainted with the original transformer architecture from the paper [Attention Is All You Need](https://arxiv.org/abs/1706.03762), you will see many parallels between the DiT block shown in the architecture overview above and the original transformer block. Nevertheless, there are some differences that we will explain here as well as a shortrefresher of the most core concepts. The corresponding code is in the class `DitBlock` in `diffusion/modules.py`.

#### AdaLN-Single
Adaptive layer normalization (AdaLN) is a wide-spread technique in image generation (original paper: [FiLM: Visual Reasoning with a General Conditioning Layer](https://arxiv.org/abs/1709.07871)). It dates back to when GANs were still the dominant model architecture for image generation and was also applied in U-Net backbones of diffusion models before naturally finding it's way into the DiT, since the transformer architecture featured normalization layers as well. In the original DiT paper, [Scalable Diffusion Models with Transformers](http://arxiv.org/abs/2212.09748), you'll find ablation studies showed that adaptive layer normalization indeed performed better than its vanilla counterpart. The attentive reader will now notice that AdaLN the previously mentioned DiT implementation differs from what they see in the architecture overview above. In the original DiT implementation, each DiT block has a separate MLP that outputs the layer normalization parameters conditioned on the timestep and context. In the figure on top we see one MLP that is generating the layer normalization parameters for all DiT blocks, while also just using the timestep as conditing. This is an important contribution of the PixArt-Alpha paper. They showed that a single MLP (hence _AdaLN-Single_) creating all layer normalization parameters doesn't impact performance and saves a substantial amount of parameters (in their case 27%!). One additional thing that AdaLN-Signle has, are  layer-specific embeddings that are added to the predicted normalization parameters. The AdaLN-Single code implementation is in the class `AdaLNSingle` in `diffusion/modules.py`.


You may ask yourself now, how we do we condition on the context then? We handle the conditioning on the context, i.e. the label embedding, via cross-attention, which will be explained in the coming section. 

#### Cross-Attention
Sometimes there are tokens that aren't direcly part of the generative process, but still contain important information that we want to make accessible to the attention mechanism, those are in our case the label embeddings. To achieve that we add the label embedding token to our attention mechanism and let other token's $K$ vector query it. Since the noised image doesn't influence the label token, hence we won't query other tokens and aggregate their information, there's no need to create a $Q$ vector for the label token.  

#### Multi-Head Attention
As we learned earlier, the attention mechanism searches relevant information and routes it between tokens. One set of neurals, one for $Q$, $K$, and $V$ respectively, is called one head. Multi-head attention is the concept of running several of those attention heads in parallel. The idea behind it is that different heads learn to focus on different pattern in the data that complement each other. After applyingthe attention heads to all tokens, the weighted average of $V$s from each head are concatinated for each token and then brought back into the original token dimension with a linear projection. 

#### Residual Connections
Residual connections are a staple of deep learning since over ten years. Origially dsigned to improve training performance of deep networks used for image-based tasks, they also started to be used extensively in other domains, such as language model architectures like transformers. They came full-circle, powering not just early diffusion backbone architectures like U-Nets but also DiTs. Residual networks originate from an observation during the early days of deep learning, where they realized that adding more layers over a certain threshold led to a deterioriation of the model's performance. That a model with more layers performs worse doesn't make sense from a cenceptual standpoint: the smaller model should be a lower bound for the reached loss, since you could reach the same performance by setting the additional layers as a linear mapping. They hypothesized that the learning objective could lead to suboptimal conditions for convergence and adapted the model architecture so that the neural layer only learns to output the difference, i.e. residual, between the input and targeted output. Check out the paper
[Deep Residual Learning for Image Recognition](https://arxiv.org/abs/1512.03385) if you are interested in the details.

## Other Design Choices
### Noise Schedule
The repository comes with three noise schedulers, a cosine, sigmoid, and linear noise scheduler, where we use by default the cosine noise scheduler. For more details on how noise schedulers work and how they compare to each other, check out `notebooks/noise_scheduling.ipynb`.

### Sampler
PixArt-Alpha uses the DPM-Solver from the paper [DPM-Solver: A Fast ODE Solver for Diffusion Probabilistic Model Sampling in Around 10 Steps](https://arxiv.org/abs/2206.00927), which is a ODE solver that is tailored to diffsion model sampling. But for simplicity sake, we chose to integrate the sampling algorithm presented in [Denoising Diffusion Implicit Models](http://arxiv.org/abs/2010.02502), which also allows us to reduce the sampling steps significantly but is much less sophisticated than the DPM-Solver, hence potentially needs more sampling steps to reach a similarly good result.

### Data Preprocessing
We use [CIFAR-10](https://www.cs.toronto.edu/~kriz/cifar.html) as our dataset of choice due to its mutually exclusive labels, signifiance in the image ML space, and relative small size, which makes it suitable for use on consumer hardware and complex enough for taining a model beyond a mere toy example. We encode the images in advance of training and save them as a latent dataset to achieve a smaller memory footprint during training.

### Optimizer
We use AdamW as our default optimizer in this code base for the simple reason that it showed to be empirically robost both in hyperparametrization and range of applications. Especially for transformers, alternative optimizers, like SGD, have shown to be brittle due to the nonstationary nature of training schedules, e.g. curriculum learning, and heterogeneous gradients of different parameter groups. AdamW does this by assigning individual learning rates for each parameter that update throughout the process, depending on the scale of the corresponding gradients. I would advise to read the original Adam paper, since it's such an essential part of modern deep learning: [Adam: A Method for Stochastic Optimization](https://arxiv.org/abs/1412.6980). Now you maybe ask yourself what's the difference between Adam and AdamW. There was a bug in the original implementation of Adam. It included a L2 weight regularization, which wasn't decoupled from the gradient normalisation. This led to uneven and overall less impactful weight regularization. AdamW is a fix to this and decoupled the adaptive learning rate from the weight decay. But since people already used Adam in other papers, other researchers still needed the old, flawed version of Adam for comparison purposes, therefore AdamW became its own optimizer. So if you ever train a model that you don't have to benchmark against an older publication, always use AdamW over Adam. 

## Key Takeaways
We took a look at the architecture and design decisions of our diffusion model and explored why certain techniques improve training and preformance of a model and where we made trade-offs for simplicity.

- The data-agnostic nature of diffusion allows us to work on them in the highly compressed latent space instead in pixels, which leads to much more computational efficiency both in training and inference.
- VAEs allow us to transform image in and out of the latent space. Since their training objective is decoupled from the one of the diffusion backbone, we can train the both independently from each other.
- Transformers are the preferred architecture for backbones in modern diffusion models. Aside of a few tweaks, the the diffusion transformer builds on the same architecture as the original design.
- Diffusion models contain additional design choices aside of the model architecture and training procedure. We discussed noise schedule, sampler, and optimizer choice. Here we mostly rely on empirical results of previous publications.  